In [1]:
from pydantic_ai.hooks.agent import ReactiveAgent
from pydantic_ai.hooks.tool import ReactiveTool, ReactiveToolset

In [2]:
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider
import os
from dotenv import load_dotenv
import asyncio
from pydantic_ai import ModelRetry  # We'll use this for denying approval

load_dotenv(dotenv_path="env/.env")

# Create the model with OpenRouter configuration
api_key = os.environ.get("OPENROUTER_API_KEY")
base_url = os.environ.get("OPENROUTER_BASE_URL")

model = OpenAIModel(
    model_name="gpt-3.5-turbo",
    provider=OpenAIProvider(base_url=base_url, api_key=api_key),
)

In [3]:
async def delete_user(user_id: str) -> str:
    """Simulates deleting a user (dangerous operation)."""
    return f"User {user_id} has been deleted"

dangerous_tool = ReactiveTool(
    name="delete_user",
    function=delete_user,
    description="Deletes a user from the system"
)

# Create an event for approval
approval_event = asyncio.Event()
approval_choice = None  # Global to store the approval choice

# Create approval UI (simulated with print and input)
async def get_approval(tool_name: str, args: dict) -> bool:
    global approval_choice
    print(f"\nAPPROVAL REQUIRED for {tool_name}")
    print(f"Arguments: {args}")
    print("Options:")
    print("1. Approve once")
    print("2. Approve for entire session")
    print("3. Deny")
    
    # Get real user input
    choice = input("Enter choice (1-3): ")
    approval_choice = choice
    approval_event.set()  # Signal that we have a choice
    
    if choice == "3":
        return False
    return True

# Add approval hook
async def check_approval(context):
    print("\nStarting approval process...")
    name = context['name']
    args = context['args']
    print(f"Tool requiring approval: {name}")
    print(f"Arguments: {args}")
    
    # Reset event
    approval_event.clear()
    print("Event cleared, waiting for approval...")
    
    # Start approval process
    approval_task = asyncio.create_task(get_approval(name, args))
    
    # Wait for approval (non-blocking)
    await approval_event.wait()
    print(f"Got approval choice: {approval_choice}")
    
    # Check the choice
    if approval_choice == "3":  # Deny
        print("Denying tool execution...")
        raise ModelRetry("Tool execution denied by user")
    elif approval_choice == "2":  # Approve for session
        print("Approving for entire session...")
        # Remove the approval hook
        dangerous_tool.on.before = None
    else:
        print("Approving once...")

# Set the approval hook
dangerous_tool.on.before = check_approval


In [4]:
# Create a reactive toolset with our dangerous tool
toolset = ReactiveToolset([dangerous_tool])

# Create an agent with the toolset
agent = ReactiveAgent(
    model=model,
    toolsets=[toolset],
    instructions="You are a system administrator."
)

# Add logging hooks
async def log_before(context):
    name = context['name']
    args = context['args']
    print(f"\nAgent hook: Tool {name} is about to run with args: {args}")

async def log_after(context):
    name = context['name']
    result = context['result']
    print(f"\nAgent hook: Tool {name} completed with result: {result}")

async def log_error(context):
    name = context['name']
    error = context['error']
    print(f"\nAgent hook: Tool {name} failed with error: {error}")

# Set the logging hooks
print("\nSetting up agent hooks...")
agent.on.before_any_tool = log_before
agent.on.after_any_tool = log_after
agent.on.any_tool_error = log_error

# Verify hooks are set
print(f"Before hook set: {agent.on.before_any_tool is not None}")
print(f"After hook set: {agent.on.after_any_tool is not None}")
print(f"Error hook set: {agent.on.any_tool_error is not None}")
print(f"Tool approval hook set: {dangerous_tool.on.before is not None}")



Setting up agent hooks...
Before hook set: True
After hook set: True
Error hook set: True
Tool approval hook set: True


In [5]:
# Example usage:
print("\nExample 1: Direct tool call")
print("Try option 1 (approve once)")
try:
    await agent.call_tool(dangerous_tool, "test_user")
except ModelRetry as e:
    print(f"\nTool denied: {e}")
except Exception as e:
    print(f"\nUnexpected error: {e}")


Example 1: Direct tool call
Try option 1 (approve once)

Agent hook: Tool delete_user is about to run with args: ('test_user',)

Starting approval process...
Tool requiring approval: delete_user
Arguments: ('test_user',)
Event cleared, waiting for approval...

APPROVAL REQUIRED for delete_user
Arguments: ('test_user',)
Options:
1. Approve once
2. Approve for entire session
3. Deny
Got approval choice: 1
Approving once...

Agent hook: Tool delete_user completed with result: User test_user has been deleted


In [6]:
print("\nExample 2: Agent run")
print("Try option 2 (approve for session)")
try:
    await agent.run("Please delete the user with ID 'another_user'")
except Exception as e:
    print(f"\nError during agent run: {e}")


Example 2: Agent run
Try option 2 (approve for session)
Node type: UserPromptNode
Node type: ModelRequestNode
Node type: CallToolsNode
Found CallToolsNode, response parts: ['ToolCallPart']
Processing part: ToolCallPart

Agent hook: Tool delete_user is about to run with args: {"user_id":"another_user"}

Starting approval process...
Tool requiring approval: delete_user
Arguments: {'user_id': 'another_user'}
Event cleared, waiting for approval...

APPROVAL REQUIRED for delete_user
Arguments: {'user_id': 'another_user'}
Options:
1. Approve once
2. Approve for entire session
3. Deny
Got approval choice: 1
Approving once...

Agent hook: Tool delete_user completed with result: User another_user has been deleted


In [7]:
print("\nExample 3: Direct tool call")
print("Try option 3 (deny)")
try:
    await agent.call_tool(dangerous_tool, "third_user")
except ModelRetry as e:
    print(f"\nTool denied: {e}")
except Exception as e:
    print(f"\nUnexpected error: {e}")


Example 3: Direct tool call
Try option 3 (deny)

Agent hook: Tool delete_user is about to run with args: ('third_user',)

Starting approval process...
Tool requiring approval: delete_user
Arguments: ('third_user',)
Event cleared, waiting for approval...

APPROVAL REQUIRED for delete_user
Arguments: ('third_user',)
Options:
1. Approve once
2. Approve for entire session
3. Deny
Got approval choice: 1
Approving once...

Agent hook: Tool delete_user completed with result: User third_user has been deleted


In [8]:
print("\nExample 4: Agent run")
print("If you approved for session in Example 2, this should not require approval")
try:
    await agent.run("Please delete the user with ID 'fourth_user'")
except Exception as e:
    print(f"\nError during agent run: {e}")


Example 4: Agent run
If you approved for session in Example 2, this should not require approval
Node type: UserPromptNode
Node type: ModelRequestNode
Node type: CallToolsNode
Found CallToolsNode, response parts: ['ToolCallPart']
Processing part: ToolCallPart

Agent hook: Tool delete_user is about to run with args: {"user_id":"fourth_user"}

Starting approval process...
Tool requiring approval: delete_user
Arguments: {'user_id': 'fourth_user'}
Event cleared, waiting for approval...

APPROVAL REQUIRED for delete_user
Arguments: {'user_id': 'fourth_user'}
Options:
1. Approve once
2. Approve for entire session
3. Deny
Got approval choice: 1
Approving once...

Agent hook: Tool delete_user completed with result: User fourth_user has been deleted
